In [3]:
import random
random.seed(42)

# Pick 5 random numbers from 0 to 255
random_numbers = [random.randint(0, 255) for _ in range(5)]
print(random_numbers)


[57, 12, 140, 125, 114]


In [6]:
import pandas as pd

# Read the CSV file
csv_path = "/users/dkang33/arithmetic-reasoning-causality/experiments/token_intervention/output/GPT-OSS_stepwise/generation_h.csv"
df = pd.read_csv(csv_path)

# Group rows based on whether they share both 'base_1_num' and 'base_2_num'
grouped = df.groupby(['base_1_num', 'base_2_num'])

# Pick groups with indices specified by random_numbers.
selected_groups = [group for idx, group in enumerate([group for _, group in grouped]) if idx in random_numbers]

print(selected_groups)

[     base_1_digits base_2_digits  base_1_num  base_2_num  base_sum  \
1332     [1, 1, 2]     [1, 5, 6]         112         156       268   
1333     [1, 1, 2]     [1, 5, 6]         112         156       268   
1334     [1, 1, 2]     [1, 5, 6]         112         156       268   
1335     [1, 1, 2]     [1, 5, 6]         112         156       268   
1336     [1, 1, 2]     [1, 5, 6]         112         156       268   
1337     [1, 1, 2]     [1, 5, 6]         112         156       268   
1338     [1, 1, 2]     [1, 5, 6]         112         156       268   
1339     [1, 1, 2]     [1, 5, 6]         112         156       268   
1340     [1, 1, 2]     [1, 5, 6]         112         156       268   
1341     [1, 1, 2]     [1, 5, 6]         112         156       268   
1342     [1, 1, 2]     [1, 5, 6]         112         156       268   
1343     [1, 1, 2]     [1, 5, 6]         112         156       268   

     source_1_digits source_2_digits  source_1_num  source_2_num  source_sum  \
1332    

In [15]:
# Concatenate all groups into a single DataFrame, selecting specific columns of interest
cols = [
    "base_1_num",
    "base_2_num",
    "source_1_num",
    "source_2_num",
    "intervention_id",
    "base_sum",
    "counterfactual_sum",
    "base_before",
    "source_number",
    "generated_text",
]
selected_df = pd.concat(selected_groups)[cols].rename(columns={"base_sum": "factual_sum", "base_before": "before_intervention", "source_number": "intervention", "generated_text": "generated_reasoning"})
selected_df["before_intervention"] = selected_df["before_intervention"].apply(lambda x: x.split("<|channel|>analysis<|message|>")[-1] if isinstance(x, str) else x)
split_strings = selected_df["generated_reasoning"].apply(
    lambda x: x.split("<|end|><|start|>assistant<|channel|>final<|message|>") if isinstance(x, str) else [x, None]
)
selected_df["generated_reasoning"] = split_strings.apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
selected_df["generated_output"] = split_strings.apply(lambda x: x[1].replace("<|return|>","") if isinstance(x, list) and len(x) > 1 else None)


print(selected_df)

      base_1_num  base_2_num  source_1_num  source_2_num  intervention_id  \
1332         112         156           112           756                8   
1333         112         156           112           756               10   
1334         112         156           112           756               12   
1335         112         156           112           756               14   
1336         112         156           112           756               18   
1337         112         156           112           756               19   
1338         112         156           112           756               20   
1339         112         156           112           756               22   
1340         112         156           112           756               23   
1341         112         156           112           756               25   
1342         112         156           112           756               26   
1343         112         156           112           756               27   

In [16]:
selected_df.to_csv(
    "/users/dkang33/arithmetic-reasoning-causality/experiments/token_intervention/output/GPT-OSS_stepwise/generation_h_filtered_for_manual_annotation.csv",
    index=False,
)
